In [0]:
CREATE WIDGET TEXT academic_year DEFAULT "202526";
CREATE WIDGET TEXT ilr_snapshot DEFAULT "SN06";
CREATE WIDGET TEXT snapshot DEFAULT "06";
CREATE WIDGET TEXT source_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";    
CREATE WIDGET TEXT output_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";
CREATE WIDGET TEXT source_schema DEFAULT "fe_skills_dev";                                       
CREATE WIDGET TEXT output_schema DEFAULT "output_202526_SN06"

In [0]:
/***********
Demographics Data for Apprenticeships Interactive Tool
Updated by:      Alison Cooper - to include provider name rather than learner lad, and now to have 'low' rather than zeros for suppressed values
Quarter:         Q2 (August to January) 2026
Snapshot:        6  
Approx run time: 1-2 mins
Rows:			
***********/

--Update
--Demographic info fields sex, age group  ethnicity_major and lldd.
--MT 27/02/2024

--update 09/12/25 include 4 years - 3 full plus latest

--Select latest IFA routes data
CREATE OR REPLACE TEMPORARY VIEW Routes_IFA as
SELECT 
std_fwk_name as std_fwk_name_routes,
std_lars_code
FROM  catalog_40_copper_proj_fe_skills_statistics_dev.ref.routes_ifa 
WHERE Snapshot = ${snapshot}  AND academic_year = ${academic_year} ;


--Select and define fields and join on routes data
CREATE OR REPLACE TEMPORARY VIEW APPS AS
SELECT 
  CASE 
    WHEN a.academic_year = ${academic_year} THEN
      CASE 
        WHEN ${snapshot} = 4 THEN CONCAT(a.academic_year, ' (Aug to Oct)')
        WHEN ${snapshot} = 6 THEN CONCAT(a.academic_year, ' (Aug to Jan)')
        WHEN ${snapshot} = 10 THEN CONCAT(a.academic_year, ' (Aug to Apr)')
        ELSE a.academic_year
      END
    ELSE a.academic_year
  END AS `year`,
age_summary as age_group,
sex,
ethnicity_major,
lldd,
name_with_ukprn as provider_name,
starts_sr as starts,
achievements_sr as achievements,
enrolments_sr as enrolments

FROM catalog_40_copper_proj_fe_skills_statistics_dev.fe_skills_dev.vw_apprenticeship_start_ach_il_ees a

LEFT JOIN Routes_IFA r 							on a.std_fwk_flag = 'Standard' and a.std_fwk_code = r.std_lars_code

WHERE (snapshot = 14 AND  a.academic_year  IN (${academic_year} -303, ${academic_year} - 202, ${academic_year} - 101)) OR (snapshot = ${snapshot} AND  a.academic_year  = ${academic_year}) ;


In [0]:
--Calculate measures and group data, and format year*
--*ie place a /(solidus) after the first 4 characters, so that date appears as, for example, 2022/23 rather than 202223

CREATE OR REPLACE TEMPORARY VIEW APPS2 AS
SELECT 
concat(substring(`year`,1,4) , '/' , substring(`year`,5,22)) as `year`,
coalesce(age_group,'Total') as age_group,
coalesce(sex,'Total') as sex,
coalesce(ethnicity_major,'Total') as ethnicity_major,
coalesce(case when lldd = 'Unknown' then 'LLDD - Unknown' else lldd end, 'Total') as lldd,
coalesce(provider_name,'Total (All providers)') as provider_name,
case when sum(starts) < 5 then 'low' else cast(round(sum(starts), -1) as varchar(8)) end as starts,
case when sum(achievements) < 5 then 'low' else cast(round(sum(achievements), -1) as varchar(8)) end as achievements
FROM APPS 
group by `year`,cube(age_group,sex,ethnicity_major,lldd,provider_name) ;

--Select * from APPS2

In [0]:


-- Need to stuff the matrix so that for a provider in a year where there are any starts/achievements
--then need all the categories
--Where they are empty will then be set to low

--Create frame

CREATE OR REPLACE TEMPORARY VIEW frame AS
select 
`year`,
provider_name,
age_group,
sex,
ethnicity_major,
lldd
from 

(

(Select
distinct provider_name ,`year`
from APPS2) as prov

cross join 

(select 'Total' as age_group
union all
select 'Under 19' as age_group
union all
select '19-24' as age_group
union all
select '25+' as age_group) as age

cross join 

(select 'Total' as sex
union all
select 'Male' as sex
union all
select 'Female' as sex) as sex

cross join 

(select 'Total' as ethnicity_major
union all
select 'White' as ethnicity_major
union all
select 'Black / African / Caribbean / Black British' ethnicity_major
union all
select 'Asian / Asian British' as ethnicity_major
union all
select 'Mixed / Multiple ethnic groups' as ethnicity_major
union all
select 'Other ethnic group' as ethnicity_major
union all
select 'Unknown' as ethnicity_major) as eth

cross join 

(select 'Total' as lldd
union all
select 'No learning difficulty / disability' as lldd
union all
select 'With learning difficulty / disability' as lldd
union all
select 'LLDD - Unknown' as lldd) as lldd

)

where
(age_group not IN ('Total') and sex = 'Total' and ethnicity_major = 'Total' and lldd = 'Total') or 
(age_group = 'Total' and sex  not IN ('Total') and ethnicity_major = 'Total' and lldd = 'Total') or 
(age_group = 'Total' and sex = 'Total' and ethnicity_major  not IN ('Total') and lldd = 'Total') or 
(age_group = 'Total' and sex = 'Total' and ethnicity_major = 'Total' and lldd  not IN ('Total') ) or 
(age_group = 'Total' and sex = 'Total' and ethnicity_major = 'Total' and lldd  = 'Total'  )   ;


select * from frame

In [0]:
--joins data onto frame
--so values for all 
--ensures there is only one breakdown of the data, so not disclosive
CREATE OR REPLACE TEMPORARY VIEW ALL_FINAL AS
select
frame.*,
coalesce(apps2.starts,'low') AS starts,
coalesce(apps2.achievements,'low') AS achievements
from frame 

left join APPS2  as apps2 on 
frame.`year`              = apps2.`year`  and  
frame.provider_name       = apps2.provider_name  and
frame.age_group           = apps2.age_group and 
frame.sex                 = apps2.sex  and 
frame.ethnicity_major     = apps2.ethnicity_major  and 
frame.lldd                = apps2.lldd 

where
(frame.age_group not IN ('Total') and frame.sex = 'Total' and frame.ethnicity_major = 'Total' and frame.lldd = 'Total') or 
(frame.age_group = 'Total' and frame.sex  not IN ('Total') and frame.ethnicity_major = 'Total' and frame.lldd = 'Total') or 
(frame.age_group = 'Total' and frame.sex = 'Total' and frame.ethnicity_major  not IN ('Total') and frame.lldd = 'Total') or 
(frame.age_group = 'Total' and frame.sex = 'Total' and frame.ethnicity_major = 'Total' and frame.lldd  not IN ('Total') ) or 
(frame.age_group = 'Total' and frame.sex = 'Total' and frame.ethnicity_major = 'Total' and frame.lldd  = 'Total'  )

order by
`year` desc,
case frame.provider_name when 'Total (All providers)' then 1 else 2 end,
provider_name,
case frame.lldd when 'Total' then 1 when 'No learning difficulty / disability' then 2 when 'With learning difficulty / disability' then 3 when 
'LLDD - Unknown' then 4 end,
case frame.ethnicity_major when 'Total' then 1 when 'White' then 2 when 'Black / African / Caribbean / Black British' then 3 
when 'Asian / Asian British' then 4 when 'Mixed / Multiple ethnic groups' then 5 when 'Other ethnic group' then 6 
when 'Unknown' then 7 end,
case frame.sex when 'Total' then 1 when 'Male' then 2 when 'Female' then 3 end,
case frame.age_group when 'Total' then 1 when 'Under 19' then 2 when '19-24' then 3 when '25+' then 4 end;

select * from ALL_FINAL
